In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1.shape

(384,)

In [4]:
doc = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(doc)

In [5]:
# Compare
v1.dot(dv)

np.float32(0.323324)

In [6]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.019730454)

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

#### Generating Embeddings

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)
    
len(texts)

1401

In [10]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [11]:
from tqdm.auto import tqdm

In [12]:
# Chunk dataset into batches of 50 encode each batch
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
    
len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1401

In [13]:
# Scoring documents
import numpy as np
X = np.array(vectors)

In [14]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [15]:
scores = X.dot(v_query)
# scores = [v_query.dot(X[i]) for i in range(len(X))] (alternatively)

In [16]:
# Best match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [17]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [18]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

scores[top5]

array([0.7629411 , 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [19]:
# Alternatively
top5 = np.argsort(-scores)[:5]
scores[top5]

array([0.7629411 , 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [20]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

### VectorSearch with MinSearch

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [22]:
# Searching
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [23]:
# Filter by course
results = vindex.search(query_vector, num_results=5, filter_dict={"course" : "llm-zoomcamp"})
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

### RAG with Vector Search

In [24]:
from openai import OpenAI

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [25]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [26]:
from rag_helper import RAGBase, RAGVector

assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

In [27]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still sign up for the program. The deadline for certain components might have passed, but you can submit answers even after the deadline has closed as long as the submission form is still open. However, if you want to receive a certificate and need to submit your project while submissions are still being accepted, you should do so before they close.\n\nAs of now, no information is provided on whether or not certificates are issued for homework that was missed.'

In [31]:
vector_assistant = RAGVector(
    embedder=model,
    index = vindex,
    llm_client = openai_client
)

In [32]:
vector_assistant.rag(query)

"Yes, you can still sign up for the program, as long as you submit your project while they're still accepting submissions if you want to receive a certificate. There is no time limit mentioned in the provided context, but it's recommended to submit your project before the end of the available submission period."

In [33]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Based on the provided context, yes, you can still sign up for the program despite it having already begun. \n\n*   There is an exception where submitting a project to receive a certificate: You need to do this "while we\'re still accepting submissions". However there is no mention of when exactly submissions stop.\n\nIn terms of registration it says that you don\'t "need" a confirmation email and can start learning and submitting homework without registering. Registration is used only to gauge interest before the course date, so there\'s no indication you can sign up after it has already begun unless it meets the above requirements.\n\nIt also mentions you can run the course locally by setting up Python, uv, Jupyter, Docker and other tools if that\'s something you\'re comfortable with. It does mention you should document your setup and keep your environment reproducible but says this doesn\'t affect anything regarding enrolling for the program after it has started.'

### Vector Search with Sqlitesearch

In [34]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)

In [35]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [36]:
# Filtering by course
results = vs_index.search(
    query_vector,
    num_results=5,
    filter_dict={"course":"llm-zoomcamp"}
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [37]:
vs_index.close()